In [1]:
from ERA_Distribution_Classes_Python.Classes.ERADist import ERADist
from ERA_Distribution_Classes_Python.Classes.ERANataf import ERANataf
from ERA_Distribution_Classes_Python.Classes.FORM_HLRF import FORM_HLRF
from ERA_Distribution_Classes_Python.Classes.FORM_fmincon import FORM_fmincon
from ERA_Distribution_Classes_Python.Classes.SuS import SuS

In [2]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt 
from structure import Structure
from solver_1st import Solver1stOrder
from syst_4_model_functions import t_S_linear

## Material Properties

In [3]:
# NEW modified IPE 120 -> eta = 100% for Th.I.O.
E = 210e6 # kN/m2 
A = 1.321e-3 # m2
I = 0.2740555e-5 # m4
h = 0.12  # m
z = h/2  # m
sigma_yield = 35.5 # kN/cm2 
alpha = 1.14
M_yield = I/z * sigma_yield * 100e2 * 1.14 # kNm

In [4]:
def t_R(M_k, I=I, z=z, alpha=alpha):
    """
    Takes in the Steel Bending Strength M_k (random variable) in kN/cm2
    I in m^4
    z in m
    alpha is the plastic ratio 

    Returns the characteristic Bending Moment Resistance M_c_Rk for the given system in kNm
    """
    return I/z * M_k * 100e2 * alpha

In [5]:
# Quick Check: should give M_Rd = 18.485 kNm
print(t_R(35.5))

18.485043474999998


## System Definition

In [11]:
# Vectorized Version of the Structural response function (Better for array handling later)
t_S_vectorized = np.vectorize(t_S_linear, otypes=[float])

## Characteristic Values (for calibrating the probabilistic loads)

In [ ]:
s_k = 1.1 # snow load on ground kN/m2
q_b = 0.65 # wind pressure kN/m2
w_k = q_b * 0.8 # wind load kN/m2 with c_pe,10 = 0.8 (Area D)
m_k = 35.5 # kN/cm2 (steel yield resistance)

In [13]:
# Quick Check: should give 18.485
print(t_S_linear(l_1=s_k * 1.5, l_2=q_b * 1.5))

18.485104167077488


## Distributions

In [14]:
# Snow time-invariant part
mu_Theta_1 = 0.81
cov_Theta_1 = 0.26
sig_Theta_1 = mu_Theta_1 * cov_Theta_1
Theta_L1 = ERADist('lognormal','MOM',[mu_Theta_1, sig_Theta_1])

# Snow load on ground
mu_L1 = 1.0
cov_L1 = 0.2
sig_L1 = mu_L1 * cov_L1
L1 = ERADist('gumbel','MOM',[mu_L1,sig_L1])

In [15]:
percentile_L1 = L1.icdf(0.98)
print(f"Snow 98% Percentile: {percentile_L1}")

Snow 98% Percentile: 1.5184551765313796


In [16]:
# Wind time-invariant part
mu_Theta_2 = 0.97
cov_Theta_2 = 0.26
sig_Theta_2 = mu_Theta_2 * cov_Theta_2
Theta_L2 = ERADist('lognormal','MOM',[mu_Theta_2, sig_Theta_2])

# Wind velocity pressure
mu_L2 = 1.0 
cov_L2 = 0.14
sig_L2 = mu_L2 * cov_L2
L2 = ERADist('gumbel','MOM',[mu_L2, sig_L2])

In [17]:
percentile_L2 = L2.icdf(0.98)
print(f"Wind 98% Percentile: {percentile_L2}")

Wind 98% Percentile: 1.3629186235719657


In [18]:
# Structural Response Model Uncertainty (from JCSS Probabilistic Model Code, Part 3, Table 3.9.1)
mu_Theta_S = 1.0
cov_Theta_S = 0.1
sig_Theta_S = mu_Theta_S * cov_Theta_S
Theta_S = ERADist('lognormal','MOM',[mu_Theta_S, sig_Theta_S])  # Distribution for Moments in frames

In [19]:
# Steel bending model uncertainty
mu_Theta_M = 1.15
cov_Theta_M = 0.05
sig_Theta_M = mu_Theta_M * cov_Theta_M
Theta_M = ERADist('lognormal','MOM',[mu_Theta_M, sig_Theta_M])

# Steel yielding strength
mu_M = 1.0
cov_M = 0.05
sig_M = mu_M * cov_M
M = ERADist('lognormal','MOM',[mu_M, sig_M])

In [20]:
percentile_M = M.icdf(0.05)
print(f"Steel 5% Percentile: {percentile_M}")

Steel 5% Percentile: 0.9199464756612658


## Shifting / Scaling Distributions

In [21]:
# Snow Load on Ground, shifted to characteristic value
snow_shift = s_k / percentile_L1 # ratio of target to current percentile, by which mean and std get multiplied

mu_L1_shifted = mu_L1 * snow_shift
sig_L1_shifted = sig_L1 * snow_shift
L1_shifted = ERADist('gumbel','MOM',[mu_L1_shifted, sig_L1_shifted])

print(f"""Snow Load on Ground gets shifted by {snow_shift}""")
print(f"""Old mean: {mu_L1}; New mean: {mu_L1_shifted}""")
print(f"""Old std: {sig_L1}; New std: {sig_L1_shifted}""")
print(f"""Old 98th percentile: {L1.icdf(.98)}; New 98th percentile: {L1_shifted.icdf(.98)}""")
print(f"""Old COV: {L1.std()/L1.mean()}; New COV: {L1_shifted.std()/L1_shifted.mean()}""")

Snow Load on Ground gets shifted by 0.7244204616646898
Old mean: 1.0; New mean: 0.7244204616646898
Old std: 0.2; New std: 0.14488409233293795
Old 98th percentile: 1.5184551765313796; New 98th percentile: 1.0999999999999999
Old COV: 0.19999999999999998; New COV: 0.19999999999999996


In [22]:
# Wind velocity pressure, shifted to characteristic value
wind_shift = q_b / percentile_L2

mu_L2_shifted = mu_L2 * wind_shift
sig_L2_shifted = sig_L2 * wind_shift
L2_shifted = ERADist('gumbel','MOM',[mu_L2_shifted, sig_L2_shifted])

print(f"""Wind velocity pressure gets shifted by {wind_shift}""")
print(f"""Old mean: {mu_L2}; New mean: {mu_L2_shifted}""")
print(f"""Old std: {sig_L2}; New std: {sig_L2_shifted}""")
print(f"""Old 98th percentile: {L2.icdf(.98)}; New 98th percentile: {L2_shifted.icdf(.98)}""")
print(f"""Old COV: {L2.std()/L2.mean()}; New COV: {L2_shifted.std()/L2_shifted.mean()}""")

Wind velocity pressure gets shifted by 0.4769176888173018
Old mean: 1.0; New mean: 0.4769176888173018
Old std: 0.14; New std: 0.06676847643442226
Old 98th percentile: 1.3629186235719657; New 98th percentile: 0.65
Old COV: 0.14; New COV: 0.13999999999999999


In [23]:
# Steel bending resistance, shifted to characteristic value
steel_shift = m_k / percentile_M

mu_M_shifted = mu_M * steel_shift
sig_M_shifted = sig_M * steel_shift
M_shifted = ERADist('lognormal','MOM',[mu_M_shifted, sig_M_shifted])

print(f"""Steel bending resistance gets shifted by {steel_shift}""")
print(f"""Old mean: {mu_M}; New mean: {mu_M_shifted}""")
print(f"""Old std: {sig_M}; New std: {sig_M_shifted}""")
print(f"""Old 5th percentile: {M.icdf(0.05)}; New 5th percentile: {M_shifted.icdf(0.05)}""")
print(f"""Old COV: {M.std()/M.mean()}; New COV: {M_shifted.std()/M_shifted.mean()}""")

Steel bending resistance gets shifted by 38.58920158858403
Old mean: 1.0; New mean: 38.58920158858403
Old std: 0.05; New std: 1.9294600794292016
Old 5th percentile: 0.9199464756612658; New 5th percentile: 35.5
Old COV: 0.04999999999999947; New COV: 0.04999999999999946


## Partial Safety Factors

In [24]:
gamma_M = 1.0  # Resistance
gamma_F1 = 1.5   # Snow Load
gamma_F2 = 1.5   # Wind Load
#psi_0 = 1.0 # 0.6     # Windload

In [25]:
# Array of marginal distributions
marginal_dist = [Theta_M, M_shifted, Theta_L1, L1_shifted, Theta_L2, L2_shifted, Theta_S]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf = ERANataf(M=marginal_dist, Correlation=R_xx)

## Subset Simulation

### Design Option 1

In [26]:
# deterministic design action effect
e_d_opt1 = t_S_linear(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [27]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [28]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf, samples_return)

In [29]:
# print("Subset Simulation for Design Option 1")
# print(f"P(F) = {Pf_SuS_1}")
# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_1)
# print(f"beta = {beta}")
# print(samplesX)

---

### Design Option 2

In [30]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S_linear(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S_linear(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [31]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [32]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf, samples_return)

In [33]:
# print("Subset Simulation for Design Option 2")
# print(f"P(F) = {Pf_SuS_2}")

# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_2)
# print(f"beta = {beta}")
# # print(samplesX)

## FORM Analysis

### Design Option 1

In [34]:
def g_opt_1_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [35]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_FORM, dg=[], distr=nataf)


*scipy.optimize.minimize() with  SLSQP  Method

  11  iterations... Reliability index =  3.3409548448079  --- Failure probability =  0.00041745400307083806 




In [36]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1_FORM(x_star)}")

u_star = [-0.49949595 -0.49949594  0.06127882  0.04459698  2.49607661  1.85260799
  0.99734051]
x_star = [ 1.12025271 37.59100665  0.79631939  0.70646347  1.77754036  0.62526353
  1.09912113]
(alpha_2)^2 = [2.23523262e-02 2.23523254e-02 3.36418300e-04 1.78184379e-04
 5.58180435e-01 3.07486358e-01 8.91139526e-02]
beta = 3.3409548448079
P(F) = 0.00041745400307083806
g(X*) = -1.935237854411298e-07


### Design Option 2

In [37]:
def g_opt_2_FORM(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [38]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_FORM, dg=[], distr=nataf)


*scipy.optimize.minimize() with  SLSQP  Method

  11  iterations... Reliability index =  3.340954843014855  --- Failure probability =  0.00041745400576676575 




In [39]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2_FORM(x_star)}")

u_star = [-0.49959335 -0.49959346  0.06157294  0.04489048  2.4962802   1.85214647
  0.9975592 ]
x_star = [ 1.12024726 37.59082347  0.79637929  0.70650224  1.77763292  0.62520875
  1.09914511]
(alpha_2)^2 = [2.23610445e-02 2.23610542e-02 3.39655434e-04 1.80537440e-04
 5.58271494e-01 3.07333177e-01 8.91530371e-02]
beta = 3.340954843014855
P(F) = 0.00041745400576676575
g(X*) = 1.457173581798088e-07
